# 🍓 PIPELINE HUẤN LUYỆN YOLOV8 NHẬN DIỆN BỆNH DÂU TÂY (T4 GPU)
### Đề tài NCKH: Robot Nhà Kính & Điều Phối Phun Thuốc Cục Bộ
---
**Mục tiêu:** Huấn luyện mô hình YOLOv8-Nano phân loại 5 lớp bệnh: `khoe_manh`, `dom_trang`, `vang_ua`, `kho_heo`, `chay_la`.

## 🔹 BƯỚC 1: Cài đặt thư viện & Giải nén Dataset
*Kéo thả file `dataset.zip` vào mục Files (thư mục `/content/`) bên tay trái trước khi chạy.*

In [ ]:
# 1. Cài đặt thư viện Ultralytics YOLOv8
!pip install -q ultralytics

# 2. Tự động giải nén dataset.zip
import os, zipfile

zip_candidates = [
    "/content/dataset.zip",
    "/content/drive/MyDrive/dataset.zip"
]
found_zip = None
for z in zip_candidates:
    if os.path.exists(z):
        found_zip = z
        break

if found_zip:
    print(f"📦 Đang giải nén tập dữ liệu: {found_zip}...")
    os.makedirs("/content/dataset", exist_ok=True)
    with zipfile.ZipFile(found_zip, 'r') as zip_ref:
        zip_ref.extractall("/content/dataset")
    print("✅ Đã giải nén thành công toàn bộ ảnh và nhãn vào /content/dataset!")
else:
    print("⚠️ Chưa thấy file dataset.zip. Hãy kéo thả file dataset.zip vào cột Files bên trái rồi chạy lại ô này!")

## 🔹 BƯỚC 2: Cấu hình Data YAML 5 Lớp Bệnh

In [ ]:
data_yaml_content = """
path: /content/dataset
train: images/train
val: images/val
test: images/test
nc: 5
names:
  0: khoe_manh
  1: dom_trang
  2: vang_ua
  3: kho_heo
  4: chay_la
"""
os.makedirs("/content/dataset", exist_ok=True)
with open("/content/dataset/data.yaml", "w", encoding="utf-8") as f:
    f.write(data_yaml_content.strip())

print("📄 Cấu hình data.yaml 5 lớp bệnh đã sẵn sàng!")
!cat /content/dataset/data.yaml

## 🔹 BƯỚC 3: Huấn Luyện YOLOv8 (Bảo Tồn Sắc Tố Màu Bệnh)

In [ ]:
import torch
from ultralytics import YOLO

print(f"🚀 Thiết bị đang sử dụng: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Khởi tạo mô hình YOLOv8-Nano
model = YOLO("yolov8n.pt")

# Tiến hành huấn luyện
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=60,
    batch=16,
    imgsz=640,
    device=0 if torch.cuda.is_available() else 'cpu',
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    patience=15,
    # ⚠️ Khóa cứng kênh màu (hsv_h=0.0) để bảo vệ sắc tố bệnh dâu tây
    hsv_h=0.0,
    hsv_s=0.1,
    hsv_v=0.2,
    degrees=10.0,
    fliplr=0.5,
    flipud=0.0,
    project="/content/runs",
    name="strawberry_ai",
    exist_ok=True
)

print("\n🎉 HUẤN LUYỆN HOÀN TẤT THÀNH CÔNG!")

## 🔹 BƯỚC 4: Xuất Mô Hình Sang Định Dạng ONNX & Tải Về Máy Tính

In [ ]:
import os
from ultralytics import YOLO
from google.colab import files

best_pt_path = "/content/runs/strawberry_ai/weights/best.pt"

if os.path.exists(best_pt_path):
    best_model = YOLO(best_pt_path)
    
    # Export sang ONNX (Opset 12)
    onnx_file = best_model.export(format="onnx", imgsz=640, dynamic=False, opset=12, simplify=True)
    print(f"\n🎉 File ONNX đã tạo tại: {onnx_file}")
    
    # Tự động tải file best.onnx về máy tính
    print("⬇️ Đang tải file best.onnx về máy tính của bạn...")
    files.download(onnx_file)
else:
    print(f"❌ Không tìm thấy file checkpoint: {best_pt_path}")